# Web and social media analysis
A project of social media analysis about these questions:

1. **Sentiment & framing** — Do people talk about MT as a tool, a threat, or both? How does framing differ between users (translators vs. general public vs. tech advocates)?
2. **Professional impact discourse** — How do working translators and localizers discuss MT's effect on their careers, rates, and job availability?
3. **Utility vs. displacement tension** — Do people acknowledge MT's utility while also lamenting professional erosion, or are these largely separate conversations?

## Setup

In [1]:
!pip -q install pandas atproto python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import time
from datetime import datetime, timezone, timedelta
import pandas as pd

from atproto import Client, models
from atproto_client.exceptions import RequestException, InvokeTimeoutError, NetworkError
from dotenv import load_dotenv
import os

## Data gathering

Use BlueSky social media, their API to fetch posts, threads, users and crawl to get connections.

In [3]:
# SearchPostsResponse
# │
# ├── posts: List[PostView]
# │   │
# │   └── PostView
# │       ├── uri          # unique ID of the post (e.g. at://did:plc:.../post/abc123)
# │       ├── cid          # content hash
# │       ├── like_count
# │       ├── repost_count
# │       ├── reply_count
# │       │
# │       ├── author: ProfileViewBasic
# │       │   ├── handle       # e.g. "janedoe.bsky.social"
# │       │   ├── display_name
# │       │   └── description  # ← this is the bio, useful for user labeling
# │       │
# │       └── record: Record   # ← the actual post content
# │           ├── text         # ← the post text you want
# │           ├── created_at   # timestamp
# │           └── reply: Optional
# │               ├── parent.uri   # uri of the post this replies to
# │               └── root.uri     # uri of the thread root
# │
# └── cursor   # pagination token — pass this to get the next page of results

Data gathering pipeline:

1. Search keywords → seed posts
2. Crawl threads of each seed post → more posts + reply structure (needed for graph!)
3. Crawl authors of relevant posts → more posts from same users
4. Deduplicate everything → final dataset

In [4]:
from atproto import Client

load_dotenv()
client = Client()

client.login(os.getenv("BSKY_HANDLE"), os.getenv("BSKY_PASSWORD"))

ProfileViewDetailed(did='did:plc:azbnjczlamokwskrcuwdks7m', handle='linda-sta.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, germ=None, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:azbnjczlamokwskrcuwdks7m/bafkreigg7wsdyxfvak4zzxtruqa4ow45jurr4uugjpdhcpdv44dh2bsnvm', banner=None, created_at='2026-04-06T10:10:10.449Z', debug=None, description=None, display_name='', followers_count=0, follows_count=1, indexed_at='2026-04-06T10:10:30.756Z', joined_via_starter_pack=None, labels=[], pinned_post=None, posts_count=0, pronouns=None, status=None, verification=None, viewer=ViewerState(activity_subscription=None, blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=No

In [5]:
results = client.app.bsky.feed.search_posts({'q': 'machine translation', 'lang': 'en', 'limit': 5})
post = results.posts[0]
print(post.author)  # prints the full object to explore

did='did:plc:ab5f2ega2vb6jjmjaum5apur' handle='boulderguy.bsky.social' associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=ProfileAssociatedChat(allow_incoming='none', allow_group_invites=None, py_type='app.bsky.actor.defs#profileAssociatedChat'), feedgens=None, germ=None, labeler=None, lists=None, starter_packs=None, py_type='app.bsky.actor.defs#profileAssociated') avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:ab5f2ega2vb6jjmjaum5apur/bafkreieuncds6ajnboq5cuz4en4ov3viekcnpsp5qp2tkzc6pug2g7wpzy' created_at='2023-08-13T04:13:40.308Z' debug=None display_name='Gruntled old man' labels=[] pronouns=None status=None verification=None viewer=ViewerState(activity_subscription=None, blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=None, muted=False, muted_by_list=None, py_type='app.bsky.a

In [6]:
def call_with_retries(callable_fn, *args, max_retries: int = 10, **kwargs):
    for attempt in range(max_retries):
        try:
            return callable_fn(*args, **kwargs)
        except RequestException as e:
            resp = getattr(e, "response", None)
            status = getattr(resp, "status_code", None)
            headers = getattr(resp, "headers", {}) or {}

            if status == 429:
                reset = headers.get("ratelimit-reset") or headers.get("RateLimit-Reset")
                retry_after = headers.get("retry-after") or headers.get("Retry-After")

                if reset:
                    wait_s = max(1, int(reset) - _now_utc_ts()) + 1
                elif retry_after:
                    wait_s = int(float(retry_after))
                else:
                    wait_s = min(60, 2 ** attempt)

                print(f"[429] rate limited, sleeping {wait_s}s")
                time.sleep(wait_s)
                continue

            wait_s = min(10, 2 ** attempt)
            print(f"[{status}] request failed, sleeping {wait_s}s")
            time.sleep(wait_s)

    raise RuntimeError("Too many retries / repeated failures.")

In [7]:
# Facet extraction: hashtags, mentions, links

def extract_facets(record) -> dict:
    """Extract hashtags, mentions, links from a post record (if facets exist)."""
    hashtags = []
    mentions = []
    links = []

    facets = getattr(record, "facets", None) or []
    for facet in facets:
        features = getattr(facet, "features", None) or []
        for feat in features:
            ftype = getattr(feat, "py_type", None) or getattr(feat, "type", None)

            if ftype == "app.bsky.richtext.facet#tag":
                tag = getattr(feat, "tag", None)
                if tag:
                    hashtags.append(tag)
            elif ftype == "app.bsky.richtext.facet#mention":
                did = getattr(feat, "did", None)
                if did:
                    mentions.append({"did": did})
            elif ftype == "app.bsky.richtext.facet#link":
                uri = getattr(feat, "uri", None)
                if uri:
                    links.append(uri)

    return {"hashtags": hashtags, "mentions": mentions, "links": links}

In [8]:
# Search function with time window and facets extraction
def _parse_dt_utc(iso_str: str) -> datetime:
    dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)

def search_posts_time_window(
    client,
    query: str = None,
    tag: str = None,
    since_iso: str = None,
    until_iso: str = None,
    max_posts: int = 1000,
    page_size: int = 50,
    polite_sleep: float = 0.25,
    print_every_page: bool = False,
):
    cursor = None
    rows = []
    page = 0

    while len(rows) < max_posts:
        page += 1

        params = models.AppBskyFeedSearchPosts.Params(
            q=query if query else tag,
            tag=[tag] if tag else None,
            sort="latest",
            since=since_iso,
            until=until_iso,
            limit=min(page_size, max_posts - len(rows)),
            cursor=cursor,
        )

        res = call_with_retries(client.app.bsky.feed.search_posts, params)
        cursor = res.cursor
        posts = res.posts or []

        if not posts:
            if print_every_page:
                print(f"[page {page}] no posts, stopping.")
            break

        if print_every_page:
            dts = []
            for p in posts:
                created = getattr(getattr(p, 'record', None), 'created_at', None)
                if created:
                    dts.append(_parse_dt_utc(created))
            if dts:
                print(
                    f"[page {page}] newest={max(dts).isoformat()}  oldest={min(dts).isoformat()}  "
                    f"collected={len(rows)}  cursor={'yes' if cursor else 'no'}"
                )

        for p in posts:
            rec = getattr(p, "record", None)
            created = getattr(rec, "created_at", None)
            if not created:
                continue

            created_dt = _parse_dt_utc(created).strftime("%Y-%m-%dT%H:%M:%S.000Z")
            if not (since_iso <= created_dt < until_iso):
                continue

            author = getattr(p, "author", None)

            # Extract facets
            facets_data = extract_facets(rec)

            rows.append({
                "created_at": created_dt,
                "uri": p.uri,
                "cid": getattr(p, "cid", None),
                "text": getattr(rec, "text", None),
                "author_handle": getattr(author, "handle", None),
                "author_did": getattr(author, "did", None),
                "reply_count": getattr(p, "reply_count", None),
                "like_count": getattr(p, "like_count", None),
                "repost_count": getattr(p, "repost_count", None),
                "quote_count": getattr(p, "quote_count", None),
                "hashtags": facets_data["hashtags"],
                "mentions": facets_data["mentions"],
                "links": facets_data["links"],
            })

            if len(rows) >= max_posts:
                break

        if cursor is None:
            if print_every_page:
                print(f"[page {page}] cursor is None, stopping.")
            break

        time.sleep(polite_sleep)

    df = pd.DataFrame(rows)
    if not df.empty:
        df["created_at"] = pd.to_datetime(df["created_at"], utc=True, errors="coerce")
        df = df.sort_values("created_at", ascending=True).reset_index(drop=True)
    return df

In [9]:
def get_followers_list(client, actor: str, max_total: int = 300, page_size: int = 100, polite_sleep: float = 0.6):
    cursor = None
    out = []
    page = 0
    while len(out) < max_total:
        page += 1
        params = models.AppBskyGraphGetFollowers.Params(
            actor=actor,
            limit=min(page_size, max_total - len(out)),
            cursor=cursor
        )
        res = call_with_retries(client.app.bsky.graph.get_followers, params)
        cursor = res.cursor
        followers = res.followers or []

        out.extend([{
            "handle": getattr(f, "handle", None),
            "did": getattr(f, "did", None),
            "display_name": getattr(f, "display_name", None),
            "description": getattr(f, "description", None),
        } for f in followers])

        print(f"[followers page {page}] downloaded={len(out)} cursor={'yes' if cursor else 'no'}")
        if cursor is None or not followers:
            break
        time.sleep(polite_sleep)
    return out


def get_following_list(client, actor: str, max_total: int = 300, page_size: int = 100, polite_sleep: float = 0.6):
    cursor = None
    out = []
    page = 0
    while len(out) < max_total:
        page += 1
        params = models.AppBskyGraphGetFollows.Params(
            actor=actor,
            limit=min(page_size, max_total - len(out)),
            cursor=cursor
        )
        res = call_with_retries(client.app.bsky.graph.get_follows, params)
        cursor = res.cursor
        follows = res.follows or []

        out.extend([{
            "handle": getattr(f, "handle", None),
            "did": getattr(f, "did", None),
            "display_name": getattr(f, "display_name", None),
            "description": getattr(f, "description", None),
        } for f in follows])

        print(f"[following page {page}] downloaded={len(out)} cursor={'yes' if cursor else 'no'}")
        if cursor is None or not follows:
            break
        time.sleep(polite_sleep)
    return out

In [10]:
# Time window: last 12 months
UNTIL_ISO = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
SINCE_ISO = (datetime.now(timezone.utc) - timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%S.000Z")

print(f"Collecting from {SINCE_ISO} to {UNTIL_ISO}")

In [11]:
# Define queries and hastags to start from
TEXT_QUERIES = [
    '"machine translation"',  
    '"AI translation"',
    '"post-editing"',
    '"human translation"',
    '"translation quality"',
    '"translator jobs"',
    '"translation industry"',
    'DeepL',                   
    'MTPE',
]

COMPOUND_QUERIES = [
    '"post-editing" AI',
    '"AI translation" threat',
    'scraped translation',
    '"machine translation" slop',
    '"human translator" AI',
    '"translation industry" AI',
]

HASHTAG_QUERIES = [
    'machinetranslation',
    'MTPE',
    'localization',
    'l10n',
    'SlowTranslation',
    'HumanTranslation',
    'PauseAI',
    'NoAI',
]

In [12]:
all_dfs = []

for q in TEXT_QUERIES:
    df_q = search_posts_time_window(
        client, query=q,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=50, page_size=25, polite_sleep=0.35,
    )
    df_q['query'] = q
    all_dfs.append(df_q)

for tag in HASHTAG_QUERIES:
    df_t = search_posts_time_window(
        client, tag=tag,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=50, page_size=25, polite_sleep=0.35,
    )
    df_t['query'] = f'#{tag}'
    all_dfs.append(df_t)

for q in COMPOUND_QUERIES:
    df_q = search_posts_time_window(
        client, query=q,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=50, page_size=25, polite_sleep=0.35,
    )
    df_q['query'] = q
    all_dfs.append(df_q)

df_seeds = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset='uri').reset_index(drop=True)
print(f"\nTotal seed posts: {len(df_seeds)}")
print(f"Unique authors: {df_seeds['author_handle'].nunique()}")
print(f"\nBreakdown:\n{df_seeds['query'].value_counts()}")


Total seed posts: 837
Unique authors: 651

Breakdown:
query
"machine translation"         50
"translation industry"        50
#machinetranslation           50
#PauseAI                      50
#NoAI                         50
"human translator" AI         50
"AI translation"              49
"human translation"           49
"translation quality"         49
MTPE                          49
#localization                 49
#l10n                         49
"post-editing"                48
DeepL                         48
"post-editing" AI             44
"translation industry" AI     29
"machine translation" slop    24
scraped translation           21
"translator jobs"             19
#HumanTranslation              7
"AI translation" threat        3
Name: count, dtype: int64


In [13]:
df_seeds.to_csv('data/seeds.csv', index=False)
print("Saved seeds.csv")

Saved seeds.csv
